# Exp10 Part 1/5: Baseline (No Steering, No Rep Penalty)
max_new_tokens=200, greedy, 500 test samples

In [ ]:
!pip install -q bitsandbytes accelerate transformers torch bert-score tqdm
import os,json,glob,random,time,gc,datetime
import numpy as np, torch
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
OUTPUT_DIR='/kaggle/working'
CONDITION='Baseline'
MAX_NEW=200; LOG_EVERY=25; CKPT_EVERY=100
print(f'Condition: {CONDITION}, max_new_tokens={MAX_NEW}')

In [ ]:
# Load Data
DATA_FN='vietnamese_medical_halueval_15k_specialized.json'
for p in [f'/kaggle/input/**/{DATA_FN}',f'/kaggle/input/{DATA_FN}',f'data/{DATA_FN}',f'./{DATA_FN}']:
    m=glob.glob(p,recursive=True)
    if m: data_path=m[0]; break
else: raise FileNotFoundError(DATA_FN)
with open(data_path,'r',encoding='utf-8') as f: raw=json.load(f)
sh=list(raw); random.seed(SEED); random.shuffle(sh)
n=len(sh); nt=int(n*0.70); nv=int(n*0.15)
test_sub=sh[nt+nv:][:500]
print(f'Test subset: {len(test_sub)}')

In [ ]:
# Load Model
from transformers import AutoModelForCausalLM,AutoTokenizer,BitsAndBytesConfig
MODEL='Qwen/Qwen2.5-7B-Instruct'
bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.bfloat16,bnb_4bit_use_double_quant=True)
tokenizer=AutoTokenizer.from_pretrained(MODEL,trust_remote_code=True)
tokenizer.padding_side='left'
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL,quantization_config=bnb,device_map='auto',trust_remote_code=True)
model.eval()
total_params=sum(p.numel() for p in model.parameters())
peak_load=torch.cuda.max_memory_allocated()/(1024**3) if torch.cuda.is_available() else 0
print(f'Loaded {MODEL}, {total_params/1e9:.2f}B params, GPU:{peak_load:.2f}GB')

In [ ]:
# Eval Engine
from bert_score import score as bs_fn
PROMPT_T="""Dua vao ngu canh y hoc sau day, hay tra loi cau hoi:
Ngu canh: {context}
Cau hoi: {question}
Tra loi: """

class Hook:
    def __init__(s,li,v,a=18.0,K=16,d='hard'):
        s.li=li;s.v=v;s.a=a;s.K=K;s.d=d;s.t=0;s.h=None
    def ea(s,t):
        if s.K>=999:return s.a
        if t>=s.K:return 0.0
        if s.d=='hard':return s.a
        elif s.d=='linear':return s.a*(1.0-t/s.K)
        return s.a
    def fn(s,mod,inp,out):
        a=s.ea(s.t)
        if a>0:
            if isinstance(out,tuple):
                h=out[0];v=s.v.to(h.device).to(h.dtype);h[:,-1,:]=h[:,-1,:]+a*v;out=(h,)+out[1:]
            else: v=s.v.to(out.device).to(out.dtype);out[:,-1,:]=out[:,-1,:]+a*v
        s.t+=1;return out
    def reg(s,m):s.t=0;s.h=m.model.layers[s.li].register_forward_hook(s.fn)
    def rm(s):
        if s.h:s.h.remove();s.h=None

def run(model,tok,recs,v_vec,alpha,K,decay,max_new,rep_pen=1.0,layer=8,name=''):
    gt,ra,na=[],[],[]
    lats,toks,eoss,r4s=[],[],[],[]
    N=len(recs);t0=time.time()
    if torch.cuda.is_available():torch.cuda.reset_peak_memory_stats()
    print(f'\n{"="*70}\n[{name}] {N} samples, max={max_new}, rep_pen={rep_pen}, a={alpha}, K={K}, d={decay}\n[{name}] {datetime.datetime.now().strftime("%H:%M:%S")}\n{"="*70}')
    for i,rec in enumerate(recs):
        ctx=rec.get('knowledge_context',rec.get('context',''))
        q=rec['question']
        inp=tok(PROMPT_T.format(context=ctx,question=q),return_tensors='pt').to('cuda')
        hk=None
        if v_vec is not None:
            hk=Hook(layer,v_vec,alpha,K,decay);hk.reg(model)
        tt=time.time()
        with torch.no_grad():
            oi=model.generate(**inp,max_new_tokens=max_new,do_sample=False,repetition_penalty=rep_pen)
        lat=time.time()-tt
        if hk:hk.rm()
        g=oi[0][inp.input_ids.shape[1]:]
        tx=tok.decode(g,skip_special_tokens=True)
        eos=bool(len(g)>0 and g[-1].item()==tok.eos_token_id)
        w=tx.split()
        r4=((1.0-len(set(tuple(w[j:j+4]) for j in range(len(w)-3)))/max(len(w)-3,1))*100) if len(w)>=4 else 0.0
        gt.append(tx);ra.append(rec['right_answer']);na.append(rec['hallucinated_answer'])
        lats.append(lat);toks.append(len(g));eoss.append(eos);r4s.append(r4)
        if (i+1)%LOG_EVERY==0 or i==N-1:
            el=time.time()-t0;sp=(i+1)/el;eta=(N-i-1)/sp if sp>0 else 0
            gm=torch.cuda.memory_allocated()/(1024**3) if torch.cuda.is_available() else 0
            print(f'  [{name}] {i+1:>4}/{N} | {el/60:.1f}m | ETA {eta/60:.1f}m | {sp:.2f}s/s | lat {np.mean(lats[-LOG_EVERY:]):.1f}s | tok {np.mean(toks[-LOG_EVERY:]):.0f} | EOS {np.mean(eoss)*100:.0f}% | R4 {np.mean(r4s):.1f}% | GPU {gm:.2f}GB')
        if (i+1)%CKPT_EVERY==0:
            cp=os.path.join(OUTPUT_DIR,f'ckpt_{name}_{i+1}.json')
            with open(cp,'w',encoding='utf-8') as f:json.dump({'n':name,'done':i+1,'texts':gt.copy(),'refs':ra.copy(),'negs':na.copy(),'eos':eoss.copy(),'r4':r4s.copy()},f,ensure_ascii=False)
            print(f'  >>> CKPT: {cp}')
    pk=torch.cuda.max_memory_allocated()/(1024**3) if torch.cuda.is_available() else 0
    gtime=time.time()-t0
    print(f'\n  [{name}] Gen done: {gtime/60:.1f}m')
    print(f'  [{name}] BERTScore...')
    tb=time.time()
    _,_,bsr=bs_fn(gt,ra,lang='vi',model_type='bert-base-multilingual-cased',num_layers=9,verbose=True,batch_size=64)
    _,_,bsn=bs_fn(gt,na,lang='vi',model_type='bert-base-multilingual-cased',num_layers=9,verbose=True,batch_size=64)
    bst=time.time()-tb
    br,bn=bsr.numpy(),bsn.numpy()
    rp=(br>bn).astype(int)
    res={'name':name,'refpref_pct':float(rp.mean()*100),'refpref_n':int(rp.sum()),'total':len(rp),
         'bs_f1':float(np.mean(br)),'bs_std':float(np.std(br)),
         'rep4':float(np.mean(r4s)),'rep4_std':float(np.std(r4s)),
         'eos_pct':float(np.mean(eoss)*100),'eos_n':int(sum(eoss)),
         'lat':float(np.mean(lats)),'tok':float(np.mean(toks)),
         'peak_gpu':pk,'rep_pen':rep_pen,'gen_min':gtime/60,'bs_min':bst/60}
    print(f'\n  === RESULT [{name}] ===')
    print(f'  RefPref: {res["refpref_pct"]:.2f}% ({res["refpref_n"]}/{res["total"]})')
    print(f'  BS-F1:   {res["bs_f1"]:.4f} +/- {res["bs_std"]:.4f}')
    print(f'  Rep-4:   {res["rep4"]:.2f}% +/- {res["rep4_std"]:.2f}%')
    print(f'  EOS:     {res["eos_pct"]:.1f}% | Lat: {res["lat"]:.2f}s | Tok: {res["tok"]:.0f}')
    print(f'  GPU:     {res["peak_gpu"]:.2f}GB | Time: {(gtime+bst)/60:.1f}m')
    return res
print('Engine ready')

In [ ]:
# RUN: Baseline
result=run(model,tokenizer,test_sub,v_vec=None,alpha=0,K=0,decay='hard',
           max_new=MAX_NEW,rep_pen=1.0,layer=8,name=CONDITION)
result['model']=MODEL
result['params_B']=total_params/1e9
result['peak_load_gpu']=peak_load
out=os.path.join(OUTPUT_DIR,f'exp10_{CONDITION.lower()}.json')
with open(out,'w',encoding='utf-8') as f:json.dump(result,f,indent=2,ensure_ascii=False)
print(f'\nSaved: {out}')
print(f'Done: {datetime.datetime.now()}')